# Experiment 4 — Regularized Probability Stacking

**Question.** Can a small meta-classifier learn complementary patterns in the TF-IDF and XLM-R predictions and improve on **temperature-calibrated 50/50**? This experiment trains **no new text encoder**.

**Design.** Reconstruct the identical 8,000-row validation sample and verify archived predictions. Run **4-fold out-of-fold evaluation**: each fold fits calibration temperatures, chooses Logistic Regression regularization using **inner 3-fold CV on the training portion only**, trains the stacker, and evaluates on untouched outer validation rows. A calibrated reference is fitted on each outer training fold too. Evaluate pure stacking and conservative 25%/50% stacking blends. A bootstrap confidence interval and fold-by-fold results help judge whether the improvement is reliable.


In [1]:
from pathlib import Path
import glob, hashlib, json, warnings
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, classification_report, confusion_matrix, log_loss

SEED=42
CLASSES=['human','ai','fusion']
L2I={c:i for i,c in enumerate(CLASSES)}
EXPECTED_FP='97fed7e0aa8163e74359d495ffc4a025'
OUT=Path('/kaggle/working/experiment_4_stacking_outputs') if Path('/kaggle/working').exists() else Path('./experiment_4_stacking_outputs')
OUT.mkdir(parents=True, exist_ok=True)
TFIDF_PATH=None; XLMR_PATH=None; BASELINE_SUBMISSION_PATH=None
print('Output directory:',OUT)


Output directory: /kaggle/working/experiment_4_stacking_outputs


## 1. Data, split, and strict prediction alignment checks

In [2]:
def unique_file(name, explicit=None):
    if explicit is not None:
        p=Path(explicit)
        if not p.is_file(): raise FileNotFoundError(str(p))
        return p
    paths=sorted(set(str(p) for root in ['/kaggle/input','.'] for p in Path(root).glob('**/'+name)))
    if not paths: raise FileNotFoundError(f'{name} missing; add the relevant Kaggle input')
    if len(paths)>1: raise RuntimeError(f'Multiple {name}: {paths}. Set explicit path.')
    return Path(paths[0])

def check_probs(P,n,name):
    P=np.asarray(P,dtype=np.float64)
    if P.shape!=(n,3) or not np.isfinite(P).all() or np.any(P<0) or np.any(P>1+1e-4):
        raise ValueError(f'Invalid probabilities: {name}, shape={P.shape}')
    if not np.allclose(P.sum(axis=1),1,atol=1e-3): raise ValueError(f'Probability sums: {name}')
    P=np.clip(P,1e-12,1.)
    return P/P.sum(axis=1,keepdims=True)

def indexer(from_ids,to_ids):
    a=pd.Index([str(z) for z in from_ids]); b=pd.Index([str(z) for z in to_ids])
    if not a.is_unique or len(a)!=len(b) or set(a)!=set(b): raise ValueError('ID mismatch')
    return a.get_indexer(b)

train_path=unique_file('train.csv'); data_dir=train_path.parent
train=pd.read_csv(train_path,dtype={'id':str})
test=pd.read_csv(data_dir/'test.csv',dtype={'id':str})
template=pd.read_csv(data_dir/'sample_submission.csv',dtype={'id':str})
assert len(test)==len(template)==25000 and train.id.is_unique and test.id.is_unique
assert set(test.id)==set(template.id)
sub=train.sample(40000,random_state=SEED).reset_index(drop=True)
_,val=train_test_split(sub,test_size=.2,random_state=SEED,stratify=sub.label+'_'+sub.lang)
val=val.reset_index(drop=True)
val_ids=val.id.astype(str).tolist(); test_ids=test.id.astype(str).tolist()
y=val.label.map(L2I).to_numpy(dtype=np.int64)
fp=hashlib.md5(','.join(val_ids).encode()).hexdigest()
assert fp==EXPECTED_FP,(fp,EXPECTED_FP)
print('Validation ID fingerprint:',fp)

xp=unique_file('xlmr_probs.npz',XLMR_PATH)
tp=unique_file('tfidf_probs.npz',TFIDF_PATH)
with np.load(xp,allow_pickle=False) as z:
    assert set(['classes','va_ids','test_ids','yva','P_va','P_te']).issubset(z.files)
    assert list(map(str,z['classes']))==CLASSES
    ix=indexer(z['va_ids'],val_ids); it=indexer(z['test_ids'],test_ids)
    X=check_probs(z['P_va'][ix],len(y),'XLMR valid')
    Xt=check_probs(z['P_te'][it],len(test),'XLMR test')
    assert np.array_equal(z['yva'][ix].astype(int),y)
run_cfg=xp.parent/'run_config.json'
if run_cfg.is_file():
    config=json.loads(run_cfg.read_text()); assert config.get('best_epoch')==4
    if config.get('val_id_fingerprint'): assert config['val_id_fingerprint']==fp
else:
    print('WARNING: run_config.json not found; checking archive against recorded score.')
with np.load(tp,allow_pickle=False) as z:
    assert set(['yva','P_va','P_te']).issubset(z.files)
    if 'classes' in z.files: assert list(map(str,z['classes']))==CLASSES
    if 'va_ids' in z.files and 'test_ids' in z.files:
        ix=indexer(z['va_ids'],val_ids); it=indexer(z['test_ids'],test_ids)
        T=check_probs(z['P_va'][ix],len(y),'TFIDF valid')
        Tt=check_probs(z['P_te'][it],len(test),'TFIDF test')
        assert np.array_equal(z['yva'][ix].astype(int),y)
        tf_has_ids=True
    else:
        T=check_probs(z['P_va'],len(y),'TFIDF valid')
        Tt=check_probs(z['P_te'],len(test),'TFIDF test')
        assert np.array_equal(z['yva'].astype(int),y)
        tf_has_ids=False

if not tf_has_ids:
    bs=unique_file('sub_tfidf_baseline.csv',BASELINE_SUBMISSION_PATH)
    df=pd.read_csv(bs,dtype={'id':str})
    assert df.id.is_unique and set(df.id)==set(test_ids)
    assert np.array_equal(df.set_index('id').loc[test_ids,'label'].to_numpy(),np.array(CLASSES)[Tt.argmax(1)]), 'TF-IDF test row order mismatch'
    print('TF-IDF test order confirmed by baseline submission IDs')
ref_old=f1_score(y,(.45*X+.55*T).argmax(1),average='macro')
assert abs(ref_old-.9141)<.002 and abs(f1_score(y,X.argmax(1),average='macro')-.9049)<.002
print('Original XLM-R + TF-IDF 45/55:',round(ref_old,5),'| validation:',len(y),'| test:',len(test))


Validation ID fingerprint: 97fed7e0aa8163e74359d495ffc4a025
TF-IDF test order confirmed by baseline submission IDs
Original XLM-R + TF-IDF 45/55: 0.91409 | validation: 8000 | test: 25000


## 2. Temperature calibration and stacking features

Each outer fold fits its *own* temperatures on the 75% training portion. The stacker sees 6 calibrated class log-probabilities (three per base model); standardized features let the regularized logistic meta-classifier combine both models. Its predictions are softmax probabilities. The reference is the calibrated 50/50 average from the same fold.

In [3]:
EPS=1e-12

def temp_scale(P,t):
    Z=np.log(np.clip(P,EPS,1.0))/float(t)
    Z-=Z.max(axis=1,keepdims=True)
    Q=np.exp(Z)
    return Q/Q.sum(axis=1,keepdims=True)

def fit_temp(P,labels):
    def nll(z):
        Q=temp_scale(P,np.exp(z))
        return -np.log(np.clip(Q[np.arange(len(labels)),labels],EPS,1.)).mean()
    r=minimize_scalar(nll,bounds=(np.log(.05),np.log(20)),method='bounded')
    if not r.success: raise RuntimeError(r.message)
    return float(np.exp(r.x))

def features(Qx,Qt):
    return np.hstack([np.log(np.clip(Qx,EPS,1.)),np.log(np.clip(Qt,EPS,1.))])

def make_meta(C):
    return make_pipeline(StandardScaler(),LogisticRegression(C=float(C),max_iter=1500,random_state=SEED))

def get_fold_data(fit_idx,eval_idx):
    tx=fit_temp(X[fit_idx],y[fit_idx]); tt=fit_temp(T[fit_idx],y[fit_idx])
    xtrain=temp_scale(X[fit_idx],tx); ttrain=temp_scale(T[fit_idx],tt)
    xeval=temp_scale(X[eval_idx],tx); teval=temp_scale(T[eval_idx],tt)
    return features(xtrain,ttrain),features(xeval,teval),.5*(xeval+teval),tx,tt

def choose_C(train_idx,grid=(.03,.1,.3,1.,3.)):
    # Fully inner 3-fold CV: fit temperatures and train stacker on inner train,
    # evaluate on inner validation, all within the outer training indices.
    yy=y[train_idx]
    splitter=StratifiedKFold(n_splits=3,shuffle=True,random_state=SEED)
    scores={c:[] for c in grid}
    for ii,jj in splitter.split(np.zeros(len(train_idx)),yy):
        indices_fit=train_idx[ii]; indices_val=train_idx[jj]
        Ftr,Fva,_,_,_=get_fold_data(indices_fit,indices_val)
        for c in grid:
            model=make_meta(c).fit(Ftr,y[indices_fit])
            pred=model.predict(Fva)
            scores[c].append(f1_score(y[indices_val],pred,average='macro'))
    winner=max(grid,key=lambda c:(np.mean(scores[c]),-c))
    return winner,{str(c):round(float(np.mean(scores[c])),5) for c in grid}


## 3. Outer cross-fitting: unbiased with respect to fold-specific meta-model fitting

We report reference, pure stacking, and conservative 25% / 50% stacking blends on the **same untouched outer rows**. The hyperparameter `C` is selected inside each outer training portion.

In [4]:
outer=StratifiedKFold(n_splits=4,shuffle=True,random_state=SEED+7)
prob_oof={key:np.full((len(y),3),np.nan) for key in ['reference','stack','stack25','stack50']}
fold_results=[]
for fold,(fi,ei) in enumerate(outer.split(np.zeros(len(y)),y),1):
    Cbest,inner_scores=choose_C(fi)
    Ftr,Fev,reference,tx,tt=get_fold_data(fi,ei)
    model=make_meta(Cbest).fit(Ftr,y[fi])
    stack=model.predict_proba(Fev)
    if not np.array_equal(model[-1].classes_,np.arange(3)):
        raise ValueError('Meta-model class order unexpected')
    estimates={'reference':reference,'stack':stack,
               'stack25':.75*reference+.25*stack,
               'stack50':.5*reference+.5*stack}
    fold_row={'fold':fold,'fit_rows':len(fi),'heldout_rows':len(ei),
              'T_xlmr':round(tx,4),'T_tfidf':round(tt,4),'best_C':Cbest,
              'inner_C_scores':json.dumps(inner_scores)}
    for key,P in estimates.items():
        prob_oof[key][ei]=P
        fold_row[key+'_macro_f1']=round(f1_score(y[ei],P.argmax(1),average='macro'),5)
    fold_results.append(fold_row)
    print('Fold',fold,'| C',Cbest,'|', {k:fold_row[k+'_macro_f1'] for k in estimates})
for key,prob in prob_oof.items(): assert np.isfinite(prob).all(),key
fold_df=pd.DataFrame(fold_results)
fold_df.to_csv(OUT/'fold_metrics.csv',index=False)
display(fold_df)


Fold 1 | C 3.0 | {'reference': 0.90907, 'stack': 0.92072, 'stack25': 0.9156, 'stack50': 0.9167}
Fold 2 | C 3.0 | {'reference': 0.92308, 'stack': 0.93403, 'stack25': 0.92969, 'stack50': 0.93171}
Fold 3 | C 3.0 | {'reference': 0.91925, 'stack': 0.92938, 'stack25': 0.92702, 'stack50': 0.9257}
Fold 4 | C 3.0 | {'reference': 0.92274, 'stack': 0.93069, 'stack25': 0.92774, 'stack50': 0.93123}


,fold,fit_rows,heldout_rows,T_xlmr,T_tfidf,best_C,inner_C_scores,reference_macro_f1,stack_macro_f1,stack25_macro_f1,stack50_macro_f1
0,1,6000,2000,2.3888,0.7647,3.0,"{""0.03"": 0.92114, ""0.1"": 0.92585, ""0.3"": 0.927...",0.90907,0.92072,0.91560,0.91670
1,2,6000,2000,2.4219,0.7803,3.0,"{""0.03"": 0.91518, ""0.1"": 0.91768, ""0.3"": 0.921...",0.92308,0.93403,0.92969,0.93171
2,3,6000,2000,2.4077,0.7713,3.0,"{""0.03"": 0.91715, ""0.1"": 0.92053, ""0.3"": 0.923...",0.91925,0.92938,0.92702,0.92570
3,4,6000,2000,2.3941,0.7604,3.0,"{""0.03"": 0.91824, ""0.1"": 0.92206, ""0.3"": 0.924...",0.92274,0.93069,0.92774,0.93123


## 4. Model comparisons, uncertainty, and class-level diagnostics

In [5]:
comparison=[]
ref_pred=prob_oof['reference'].argmax(1)
ref_f1=f1_score(y,ref_pred,average='macro')
for name,P in prob_oof.items():
    pred=P.argmax(1)
    comparison.append({'model':name,'oof_macro_f1':f1_score(y,pred,average='macro'),
                       'delta_vs_reference':f1_score(y,pred,average='macro')-ref_f1,
                       'changed_predictions':int((pred!=ref_pred).sum()),
                       'corrected':int(((pred==y)&(ref_pred!=y)).sum()),
                       'broken':int(((pred!=y)&(ref_pred==y)).sum()),
                       **{f'{c}_f1':f1_score(y,pred,labels=[i],average='macro',zero_division=0) for i,c in enumerate(CLASSES)}})
comp=pd.DataFrame(comparison).sort_values('oof_macro_f1',ascending=False)
comp.to_csv(OUT/'model_comparison.csv',index=False)
display(comp.round(5))

rng=np.random.default_rng(SEED)
B=1000
for key in ['stack','stack25','stack50']:
    yh=prob_oof[key].argmax(1)
    diffs=np.empty(B)
    for i in range(B):
        ix=rng.integers(0,len(y),len(y))
        diffs[i]=f1_score(y[ix],yh[ix],average='macro')-f1_score(y[ix],ref_pred[ix],average='macro')
    print(key,'paired bootstrap ΔF1 95% interval:',np.round(np.quantile(diffs,[.025,.975]),5))
print('Reference cross-fitted macro-F1:',round(ref_f1,5))
print('NOTE: Comparing multiple methods here can overstate the selected improvement.')
print('Per-class diagnostics for reference:')
print(classification_report(y,ref_pred,target_names=CLASSES,digits=4))
print('Per-class diagnostics for 25% stacking blend:')
print(classification_report(y,prob_oof['stack25'].argmax(1),target_names=CLASSES,digits=4))


,model,oof_macro_f1,delta_vs_reference,changed_predictions,corrected,broken,human_f1,ai_f1,fusion_f1
1,stack,0.92870,0.01018,158,111,38,0.88421,0.99583,0.90605
3,stack50,0.92633,0.00780,122,85,27,0.87963,0.99542,0.90393
2,stack25,0.92500,0.00648,94,67,19,0.87854,0.99404,0.90243
0,reference,0.91852,0.00000,0,0,0,0.87189,0.98897,0.89470


stack paired bootstrap ΔF1 95% interval: [0.00675 0.01378]
stack25 paired bootstrap ΔF1 95% interval: [0.0038  0.00899]
stack50 paired bootstrap ΔF1 95% interval: [0.00479 0.01083]
Reference cross-fitted macro-F1: 0.91852
NOTE: Comparing multiple methods here can overstate the selected improvement.
Per-class diagnostics for reference:
              precision    recall  f1-score   support

       human     0.8932    0.8516    0.8719      1954
          ai     0.9795    0.9986    0.9890      3592
      fusion     0.8909    0.8985    0.8947      2454

    accuracy                         0.9320      8000
   macro avg     0.9212    0.9162    0.9185      8000
weighted avg     0.9313    0.9320    0.9315      8000

Per-class diagnostics for 25% stacking blend:
              precision    recall  f1-score   support

       human     0.8932    0.8644    0.8785      1954
          ai     0.9898    0.9983    0.9940      3592
      fusion     0.8966    0.9083    0.9024      2454

    accuracy      

## 5. Train final meta-classifier and export candidate submissions

In [6]:
all_idx=np.arange(len(y))
Cfinal,all_inner=choose_C(all_idx)
tx=fit_temp(X,y); tt=fit_temp(T,y)
Qx=temp_scale(X,tx); Qt=temp_scale(T,tt)
meta=make_meta(Cfinal).fit(features(Qx,Qt),y)
Qxt=temp_scale(Xt,tx); Qtt=temp_scale(Tt,tt)
reference_test=.5*(Qxt+Qtt)
stack_test=meta.predict_proba(features(Qxt,Qtt))
outputs={'sub_calibrated_5050_reference.csv':reference_test,
         'sub_stacking_25.csv':.75*reference_test+.25*stack_test,
         'sub_stacking_50.csv':.5*reference_test+.5*stack_test,
         'sub_stacking_pure.csv':stack_test}

def save_submission(name,P):
    df=pd.DataFrame({'id':test_ids,'label':np.array(CLASSES)[P.argmax(1)]})
    df=df.set_index('id').loc[template.id].reset_index()
    assert len(df)==25000 and df.id.is_unique and list(df.columns)==['id','label']
    assert df.label.isin(CLASSES).all()
    df.to_csv(OUT/name,index=False)
    return df

reference_sub=None
for name,P in outputs.items():
    df=save_submission(name,P)
    if reference_sub is None:
        reference_sub=df.label.to_numpy()
    print(name,'| changed from calibrated 50/50:',int((df.label.to_numpy()!=reference_sub).sum()))
np.savez_compressed(OUT/'stacking_probabilities.npz',
                    va_ids=np.array(val_ids),test_ids=np.array(test_ids),yva=y,
                    reference_oof=prob_oof['reference'],stack_oof=prob_oof['stack'],
                    stack25_oof=prob_oof['stack25'],stack50_oof=prob_oof['stack50'],
                    reference_test=reference_test,stack_test=stack_test)
params={'model':'regularized LogisticRegression over six calibrated log-probabilities',
        'seed':SEED,'validation_fingerprint':fp,'n_validation':len(y),
        'final_C':Cfinal,'inner_cv_mean_macro_f1':all_inner,
        'final_T_xlmr':tx,'final_T_tfidf':tt,
        'notes':'Outer 4-fold CV reports diagnostic results; selection across outer variants may be optimistic. All outputs are class labels on test, without test labels.'}
(OUT/'stacking_config.json').write_text(json.dumps(params,indent=2))
print('Output files:')
for p in sorted(OUT.iterdir()): print(' -',p.name)


sub_calibrated_5050_reference.csv | changed from calibrated 50/50: 0
sub_stacking_25.csv | changed from calibrated 50/50: 315
sub_stacking_50.csv | changed from calibrated 50/50: 453
sub_stacking_pure.csv | changed from calibrated 50/50: 554
Output files:
 - fold_metrics.csv
 - model_comparison.csv
 - stacking_config.json
 - stacking_probabilities.npz
 - sub_calibrated_5050_reference.csv
 - sub_stacking_25.csv
 - sub_stacking_50.csv
 - sub_stacking_pure.csv


## Interpretation

This experiment tests whether the **relationship between two models' class probabilities** adds information beyond fixed-weight averaging. Logistic Regression learns regularized, class-specific combinations of six log-probabilities; the 25% and 50% blends reduce dependence on the newly fitted meta-classifier. Meaningful improvement requires higher held-out macro-F1 and consistent folds, not merely a better score after fitting on the same 8,000 labels. If the confidence interval crosses zero or results vary substantially by fold, retain the calibrated 50/50 model (public Kaggle score 0.91719). No result is claimed until the notebook is executed.